# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: Flag high-visibility pages (≥500 impressions in March) that rank reasonably well (top 20 position) but capture fewer clicks than expected (CTR < 0.5%). This directly follows Signal 2's confirmed finding — position and CTR are related, and pages that are visible+ranked-well+low-CTR are the clearest, most defensible "something's wrong with the click-through" candidates.

I'm not using staleness as a primary driver, since Signal 1 showed only a 2-point gap between fresh and stale content — too weak to anchor a rule on.

Score: baseline_score = total_impressions * (0.5 - ctr) / 0.5 — this rewards pages that are both highly visible AND have a large CTR gap below the 0.5% threshold. A page with huge impressions and near-zero CTR scores highest; a page just barely under 0.5% scores low.

Reason code: low_ctr_visible_page

Action label: review_title_meta (the standard SEO fix for a CTR problem — title/meta description rewrite)

In [2]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN .env dosyasında bulunamadı"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Bucket by content age, compare decline rate
staleness_check = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    joined AS (
        SELECT d.*, c.content_created_date,
               DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
        FROM daily_agg d
        JOIN {TABLES['dim_content']} c ON d.content_hash_id = c.content_hash_id
        WHERE d.total_impressions > 0
    )
    SELECT
        CASE WHEN content_age_days >= 180 THEN 'stale (180+ days)' ELSE 'fresh (<180 days)' END AS age_bucket,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN imp_second_half < imp_first_half THEN 1 ELSE 0 END) / COUNT(*), 1) AS decline_pct
    FROM joined
    GROUP BY age_bucket
""").df()
staleness_check

,age_bucket,n,decline_pct
0,fresh (<180 days),83607,36.6
1,stale (180+ days),93131,38.6


Signal 1 — Staleness (content_age_days ≥ 180): Bucketed by age (fresh vs. stale, n=83,607 / n=93,131), decline rate is 36.6% for fresh content vs. 38.6% for stale content — only a 2-point gap. The direction matches the assumption behind FlyRank's stale_visible_page flag (older content is riskier), but the effect is too weak to call it a strong signal on its own. Verdict: MIXED — direction confirmed, magnitude weak.

In [4]:
ctr_position_check = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 500
    ),
    with_ctr AS (
        SELECT *,
               100.0 * total_clicks / total_impressions AS ctr,
               CASE WHEN avg_position <= 20 THEN 'top20' ELSE 'below20' END AS position_bucket
        FROM daily_agg
        WHERE avg_position > 0
    )
    SELECT
        position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(ctr), 2) AS avg_ctr,
        ROUND(100.0 * SUM(CASE WHEN ctr < 0.5 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_low_ctr
    FROM with_ctr
    GROUP BY position_bucket
""").df()
ctr_position_check

,position_bucket,n,avg_ctr,pct_low_ctr
0,top20,50764,0.32,81.1
1,below20,11160,0.13,95.0


Signal 2 — CTR vs. position (n=50,764 top20 / n=11,160 below20, filtered to ≥500 impressions): Average CTR is 0.32% for top-20 positions vs. 0.13% below position 20 — CTR does drop with worse position, confirming the direction behind FlyRank's low_ctr_visible_page flag. However, 81.1% of even top-20 pages fall under the 0.5% CTR threshold, meaning "low CTR" at this threshold is closer to the norm than the exception among high-visibility pages — a flag worth keeping, but the threshold may be too loose to isolate real outliers. Verdict: CONFIRMED — direction holds, but the specific threshold (0.5%) captures most pages, not just weak ones.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.